In [10]:
# !pip install thefuzz

In [11]:
import re, copy
import json
from thefuzz import fuzz
from thefuzz import process as process
import pandas as pd

In [12]:
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine

#QA Snowflake connection #using for Hot-fix of Flouring release
# snowflake_connection_string = "jdbc:snowflake://celanese-celanytics.privatelink.snowflakecomputing.com/?user=SVC_ADF&password=<SNOWFLAKE_PASSWORD>&db=ANALYTICS_QA&warehouse=REPORTING_WH&role=DATA_ANALYST_GST"
#Dev Snowflake connection
snowflake_connection_string = "jdbc:snowflake://celanese-celanytics.privatelink.snowflakecomputing.com/?user=SVC_ADF&password=<SNOWFLAKE_PASSWORD>&db=ANALYTICS_DEV&warehouse=REPORTING_WH&role=DATA_ANALYST_GST"
parts = snowflake_connection_string.split("//")[1].split("/")  
account = ".".join(parts[0].split('.')[:2]) 
user = parts[1].split('user=')[1].split('&')[0] 
password = parts[1].split('password=')[1].split('&')[0] 
database = parts[1].split('db=')[1].split('&')[0]
warehouse = parts[1].split('warehouse=')[1].split('&')[0]  
role = parts[1].split('role=')[1]

engine = create_engine(URL(
    account = account,
    user = user,
    password = password,
    database = database,
    schema = 'gst_curated',
    warehouse = warehouse,
    role = role
))
cur = engine.connect()

engine = create_engine(URL(
    account = account,
    user = user,
    password = password,
    database = "ANALYTICS_DEV",
    # database = "ANALYTICS_QA", #use for skipping the new brands updated in the Dev #F-hot-fix
    schema = 'gst_curated',
    warehouse = warehouse,
    role = role
))
cur_dev = engine.connect()

def read_data_from_snowflake_table(cur,query):
    df = pd.read_sql(query, cur)
    return df

In [13]:
competitor_data = read_data_from_snowflake_table(cur,"select * from competitor_data")
competitor_names = competitor_data['competitor'].unique()

In [15]:
outOfScopeData = json.load(open("../dependencies/outOfScopeData.json"))

outOfScopeDataGrades = outOfScopeData['grades']

normalized_unique_values_current = json.load(open("../dependencies/normalized_unique_values_for_grade_mapping.json"))

In [16]:
[i for i in normalized_unique_values_current["GRADE"] if "nylon" in i]

[]

In [17]:
unique_values = json.load(open("../dependencies/unique_values_22_02_24.json"))
def normalize_string(s):
    # Remove special characters and convert to lower case
    return re.sub(r'\W+', '', s).lower()


In [18]:
# def find_substring_items(list1, list2):
#     result = []
#     for item2 in list2:
#         for item1 in list1:
#             if len(item1)>3 and item1 in item2:
#                 print({item1:item2})
#                 result.append(item2)
#                 break  # Break to avoid duplicate entries of the same item2
#     return result

# items_to_remove_from_comp_grade = find_substring_items(unique_values['BRAND'], unique_values['COMPETITOR_GRADE'])


In [19]:
# unique_values['COMPETITOR_GRADE'] = [item for item in unique_values['COMPETITOR_GRADE'] if item not in items_to_remove_from_comp_grade]

unique_values['GRADE_WITHOUT_BRAND'] = list(set(unique_values['GRADE_WITHOUT_BRAND'] + ["dym", "eco-b", "eco b","eco-r", "eco r", "fit", "frhr", "hfs", "hhr",
"hrlm", "hrt", "hsl", "hslr", "hte", "htn", "htr", "ice", 
"icf", "lof", "lof2", "pcxxx", "pls/xt", 
"scxxx", "sea", "slidex", "wrf", "xap", "xap2", "xfr", "xgc"]))

# with open("../dependencies/unique_values_01_02_24.json", "w") as fp:
#     json.dump(unique_values , fp)

In [20]:
database

'ANALYTICS_DEV'

In [21]:
color_code_df = read_data_from_snowflake_table(cur_dev,"""select * from GST_CURATED.AUSP_SAP_MATERIAL_COLOR_CODE""")
# color_code_df = read_data_from_snowflake_table(cur_dev,"""select * from GST_CURATED.AUSP_SAP_MATERIAL_COLOR_CODE""")
ce_grades_with_color_code = color_code_df['sap_material_desc'].unique().tolist()
ce_grades_with_color_code = list(set([grade.lower().lstrip('zzdel').lstrip("dev ").strip() for grade in ce_grades_with_color_code]))
ce_grades_with_color_code = list(set([grade for grade in ce_grades_with_color_code if grade!='']))
ce_grades_with_color_code = list(set([normalize_string(i) for i in ce_grades_with_color_code]))

In [22]:
[i for i in ce_grades_with_color_code if "nw02" in i]

['nnw02xapcd3501blackimpa1',
 'nnw02cf2001naturalimpa1',
 'coilelecherionw0201solvlv24vdc',
 'coilelecherionw0200220v',
 'nnw02blackimpa1']

In [23]:
len(ce_grades_with_color_code)

434001

In [24]:
spt = read_data_from_snowflake_table(cur,"""select * from SPT""")
ce_grades = spt['product_cd'].unique().tolist()

In [25]:
spt_brands = [re.sub(r'\W+', '', brand).lower() for brand in list(spt['product_line'].unique())]
display(len(spt_brands))

35

In [ ]:
# validating the list of ce grades present (since for bug 203909 minlon grades were not populated in unique value list)
# df_ce_grades = pd.DataFrame(ce_grades)
# df_ce_grades.to_csv('celanese_grades_list.csv', index=False, header=False)

In [32]:
features = read_data_from_snowflake_table(cur,"""select * from FEATURE""")
all_features =list(set(features[features['property_name'].isin(['Special characteristics', 'Processing','Delivery form'])]['value_assmnt_si']))

In [33]:
col = 'brand'
feature = 'feature'
ignore_syn=[]
ignore_key=[]
positives = []
# SYNONYM data from dev database i.e. "analytics_dev". Refer to "DEFINED_NAME" and "SYNONYMS".
synonym_df = read_data_from_snowflake_table(cur_dev,"""select * from SYNONYM""")
synonym_df.columns = [x.upper() if x.islower() else x for x in synonym_df.columns]
synonym_df = synonym_df.apply(lambda x: x.str.lower())
display(synonym_df.head())
feature_synonyms = synonym_df.copy()

synonym_df = synonym_df[synonym_df['TYPE'] == col]
# feature_df = synonym_df[synonym_df['TYPE'] == feature]

synonym_df2 = synonym_df[["DEFINED_NAME", "SYNONYMS"]].copy()
synonym_df2 = synonym_df2.dropna().reset_index(drop=True)
# synonym_df2 = synonym_df2[synonym_df2['SYNONYMS'].apply(lambda x : False if ";" in x else True)].reset_index(drop=True)
# synonym_df2 = synonym_df2[synonym_df2['SYNONYMS'].apply(lambda x : True if ";" in x else False)].reset_index(drop=True)
brand_synonyms = synonym_df2.to_dict(orient='records')
brands_with_synonym = [item['DEFINED_NAME'] for item in brand_synonyms]
brand_synonyms  = {item['DEFINED_NAME']: item['SYNONYMS'] for item in brand_synonyms}
for k in brand_synonyms:
    if ";" in brand_synonyms[k]:
        brand_synonyms[k] = [x.strip() for x in  brand_synonyms[k].split(';') if x!=k]

brand_synonyms['ateva'] = ['at']
brands_with_synonym.append('ateva')


new_grade_names = []
for i in brands_with_synonym:
   brand_pattern = fr'^{re.escape(i)}\b'
   for grade in unique_values['GRADE']:
      if re.match(brand_pattern, grade):
         for s in brand_synonyms[i]:
            new_grade_names.append(re.sub(brand_pattern, s, grade))

# new_grade_names

,TYPE,DEFINED_NAME,SYNONYMS
0,auto cert,mercedes-benz,mercedes-benz; daimler; daimler-benz; daimler ...
1,auto cert,vw group,vw group; vw; bentley
2,brand,abistir,abistir
3,brand,amcel,amcel; am
4,brand,at,at


In [34]:
feature_synonyms = feature_synonyms[feature_synonyms['TYPE'] == feature]
feature_synonyms2 = feature_synonyms[["DEFINED_NAME", "SYNONYMS"]].copy()
feature_synonyms2 = feature_synonyms2.dropna().reset_index(drop=True)

In [35]:
feature_synonyms2['SYNONYM_LIST'] = feature_synonyms2['SYNONYMS'].apply(lambda x: [i.strip() for i in x.split(';') if len(i.strip())>1] if ";" in x else [i.strip() for i in x.split(',') if len(i.strip())>1] if "," in x else [x])

In [36]:
feature_synonyms2

,DEFINED_NAME,SYNONYMS,SYNONYM_LIST
0,anti-static,anti-static; static resistant; static; static ...,"[anti-static, static resistant, static, static..."
1,bio-content,bio-content; bio-based; bio; biobased; eco-b,"[bio-content, bio-based, bio, biobased, eco-b]"
2,carbon capture,carbon capture; reduce carbon capture; reduced...,"[carbon capture, reduce carbon capture, reduce..."
3,recycled content,recycled content; contains recycle; recycle; r...,"[recycled content, contains recycle, recycle, ..."
4,flame retardant,flame retardant; flameretardent; fr; flame ret...,"[flame retardant, flameretardent, fr, flame re..."
5,heat stabilized,heat stabilized or stable to heat; stable to h...,"[heat stabilized or stable to heat, stable to ..."
6,high flow,high flow; high mv; high flowability; low visc...,"[high flow, high mv, high flowability, low vis..."
7,high gloss,high gloss; enhanced gloss; highgloss,"[high gloss, enhanced gloss, highgloss]"
8,high viscosity,high viscosity; low mv; low flow; highvisc,"[high viscosity, low mv, low flow, highvisc]"
9,hydrolysis resistant,hydrolysis resistant; hydrolysis; water resist...,"[hydrolysis resistant, hydrolysis, water resis..."


In [37]:
feature_syn_dict = feature_synonyms2.to_dict(orient='records')
feature_syn_dict = {item['DEFINED_NAME']: item['SYNONYM_LIST'] for item in feature_syn_dict}

In [38]:
feature_syn_dict

{'anti-static': ['anti-static',
  'static resistant',
  'static',
  'static resistance',
  'antistatic',
  'avoid static',
  'avoid static build',
  'avoid static build up'],
 'bio-content': ['bio-content', 'bio-based', 'bio', 'biobased', 'eco-b'],
 'carbon capture': ['carbon capture',
  'reduce carbon capture',
  'reduced carbon footprint',
  'carbon footprint',
  'carbon footprint',
  'iso 14067',
  'carbon capture and utilization',
  'ccu',
  'eco-c'],
 'recycled content': ['recycled content',
  'contains recycle',
  'recycle',
  'recycled',
  'eco-r',
  'post consumer recycle',
  'post-consumer recycle',
  'pcr',
  'pir',
  'post industrial recycle',
  'post-industrial recycle'],
 'flame retardant': ['flame retardant',
  'flameretardent',
  'fr',
  'flame retarding agent',
  'flamret',
  'flamretag',
  'flame resistant',
  'flame resistance',
  'halogenated fr',
  'halogenated',
  'halogenated flame retardant'],
 'heat stabilized': ['heat stabilized or stable to heat',
  'stable to

In [39]:
for k in feature_syn_dict:
    if ";" in feature_syn_dict[k]:
        feature_syn_dict[k] = [x.strip() for x in  feature_syn_dict[k].split(';') if x!=k]
feature_syn_dict

feature_list = []

for k, v in feature_syn_dict.items():
    temp_list = []
    temp_list.append(k)
    if isinstance(v, list):
        temp_list.append(v)
    # else:
    #     feature_list.append(v)
    feature_list.append(temp_list)
feature_list

[['anti-static',
  ['anti-static',
   'static resistant',
   'static',
   'static resistance',
   'antistatic',
   'avoid static',
   'avoid static build',
   'avoid static build up']],
 ['bio-content', ['bio-content', 'bio-based', 'bio', 'biobased', 'eco-b']],
 ['carbon capture',
  ['carbon capture',
   'reduce carbon capture',
   'reduced carbon footprint',
   'carbon footprint',
   'carbon footprint',
   'iso 14067',
   'carbon capture and utilization',
   'ccu',
   'eco-c']],
 ['recycled content',
  ['recycled content',
   'contains recycle',
   'recycle',
   'recycled',
   'eco-r',
   'post consumer recycle',
   'post-consumer recycle',
   'pcr',
   'pir',
   'post industrial recycle',
   'post-industrial recycle']],
 ['flame retardant',
  ['flame retardant',
   'flameretardent',
   'fr',
   'flame retarding agent',
   'flamret',
   'flamretag',
   'flame resistant',
   'flame resistance',
   'halogenated fr',
   'halogenated',
   'halogenated flame retardant']],
 ['heat stabilize

In [40]:
for i in feature_list:
    if len(i) != 2:
        print('False')
    if not isinstance(i[0], str) and not isinstance(i[1], list):
        print('False')

    # break

In [41]:
synonym_df_new = read_data_from_snowflake_table(cur_dev,"""select * from SYNONYM""")
synonym_df_new.columns = [x.upper() if x.islower() else x for x in synonym_df_new.columns]

#### start of chemical resistance and medical certifications synonyms

In [42]:
chem_res = spt[spt['temp_c2'].str.lower().str.startswith('chemical')]

subset = chem_res[['property_id', 'property_name', 'value_assmnt_si', 'non_std_test_cond_desc']].drop_duplicates(subset='property_name').reset_index(drop=True)

df_sorted = (
    subset.assign(
        _PROPERTY_NAME_lower=subset['property_name'].str.lower(),
        _VALUE_ASSMNT_SI_lower=subset['value_assmnt_si'].str.lower()
    )
    .sort_values(['_PROPERTY_NAME_lower', '_VALUE_ASSMNT_SI_lower'])
    .drop(columns=['_PROPERTY_NAME_lower', '_VALUE_ASSMNT_SI_lower'])
    .reset_index(drop=True)
)

df_sorted.head()

,property_id,property_name,value_assmnt_si,non_std_test_cond_desc
0,340,"Acids, Formic Acid","Resistant, Long-term Exposure",23°C
1,384,"Acids, Hydrochloric Acid","Resistant, Long-term Exposure",23°C
2,252,"Acids, Strong (pH 0-3)","Resistant, Long-term Exposure",23°C
3,473,"Acids, Sulfuric Acid",No Data Available,23°C
4,261,"Acids, Weak","Limited Resistance, Short-term Exposure",23°C


In [50]:
chem_syn = synonym_df_new[synonym_df_new['TYPE'].str.lower().str.startswith('chemical')]
chem_syn

,TYPE,DEFINED_NAME,SYNONYMS
65,CHEMICAL RESISTANCE,"Acids, Formic Acid",Methanoic acid; HCOOH; Concentrated formic aci...
66,CHEMICAL RESISTANCE,"Acids, Hydrochloric Acid",Muriatic acid; HCl; Concentrated HCl (37%); Di...
67,CHEMICAL RESISTANCE,"Acids, Strong (pH 0-3)",Mineral acids; inorganic acids; Nitric acid (H...
68,CHEMICAL RESISTANCE,"Acids, Sulfuric Acid",Oil of vitriol; H₂SO₄; Concentrated sulfuric a...
69,CHEMICAL RESISTANCE,"Acids, Weak",Organic acids; mild acids; Acetic acid (vinega...
70,CHEMICAL RESISTANCE,"Alcohols, Long-Chain Alcohols (> C4)",Higher alcohols; fatty alcohols; Hexanol; Octa...
71,CHEMICAL RESISTANCE,"Alcohols, Short-Chain Alcohols (C1-C4)",Lower alcohols; light alcohols; Methanol; Etha...
72,CHEMICAL RESISTANCE,"Alkalies/Bases, Strong (pH 11-14)",Caustic solutions; alkali hydroxides; Sodium h...
73,CHEMICAL RESISTANCE,"Alkalies/Bases, Weak",Mild alkaline solutions; Sodium bicarbonate (N...
74,CHEMICAL RESISTANCE,"Coolants, Aqueous Glycol-Based Coolant",Antifreeze solutions; radiator fluids; Ethylen...


In [53]:
valid_format = json.load(open("../dependencies/chemical_resistance.json"))
valid_format

{'acids': [{'sub_category': 'formic acid',
   'resistance_level': 'limited resistance, short-term exposure',
   'synonyms': ['methanoic acid',
    'hcooh',
    'concentrated formic acid >85%',
    'dilute formic acid solutions',
    'industrial-grade formic acid',
    'formate acid',
    'hydrogen carboxylic acid',
    'formylic acid',
    'formic acid solution',
    'ant sting acid',
    'red ant acid'],
   'temperature': '23'},
  {'sub_category': 'hydrochloric acid',
   'resistance_level': 'limited resistance, short-term exposure',
   'synonyms': ['muriatic acid',
    'hcl',
    'concentrated hcl (37%)',
    'dilute hydrochloric acid (5â€“20%)',
    'hydrogen chloride solution',
    'aqueous hydrogen chloride',
    'chlorohydric acid',
    'hydronium chloride solution',
    'hydrochloride acid',
    'hydrogen chloride aqueous'],
   'temperature': '23'},
  {'sub_category': 'strong (ph 0-3)',
   'resistance_level': 'resistant, long-term exposure',
   'synonyms': ['mineral acids',
    '

In [56]:
valid_format_nested = [
    [
        category,
        [
            list(item.values()) if isinstance(item, dict) else item
            for item in entries
        ]
    ]
    for category, entries in valid_format.items()
]

valid_format_nested

[['acids',
  [['formic acid',
    'limited resistance, short-term exposure',
    ['methanoic acid',
     'hcooh',
     'concentrated formic acid >85%',
     'dilute formic acid solutions',
     'industrial-grade formic acid',
     'formate acid',
     'hydrogen carboxylic acid',
     'formylic acid',
     'formic acid solution',
     'ant sting acid',
     'red ant acid'],
    '23'],
   ['hydrochloric acid',
    'limited resistance, short-term exposure',
    ['muriatic acid',
     'hcl',
     'concentrated hcl (37%)',
     'dilute hydrochloric acid (5â€“20%)',
     'hydrogen chloride solution',
     'aqueous hydrogen chloride',
     'chlorohydric acid',
     'hydronium chloride solution',
     'hydrochloride acid',
     'hydrogen chloride aqueous'],
    '23'],
   ['strong (ph 0-3)',
    'resistant, long-term exposure',
    ['mineral acids',
     'inorganic acids',
     'nitric acid (hnoâ‚ƒ)',
     'phosphoric acid (hâ‚ƒpoâ‚„)',
     'perchloric acid (hcloâ‚„)',
     'hydrofluoric acid 

In [57]:
# Medical certification synonyms
med_cert_synonyms = synonym_df_new[synonym_df_new['TYPE'] == 'MEDICAL CERT']
med_cert_synonyms
# synonym_df['TYPE'].unique()

,TYPE,DEFINED_NAME,SYNONYMS
151,MEDICAL CERT,CYTOTOXICITY,cytotoxicity; cell toxicity; cellular toxicity...
152,MEDICAL CERT,DERMAL_IRRITATION,dermal irritation; skin irritation; skin irrit...
153,MEDICAL CERT,GENETOXICITY,genotoxicity; genetic toxicity; mutagenicity; ...
154,MEDICAL CERT,HEMOLYSIS,hemolysis; haemolysis; blood compatibility; re...
155,MEDICAL CERT,MUSCLE_IMPLANTATION,muscle implantation; intramuscular implantatio...
156,MEDICAL CERT,PHYSICOCHEMICAL_COMPLIANCE,physicochemical compliance; physical chemical ...
157,MEDICAL CERT,PYROGENICITY,pyrogenicity; pyrogen test; pyrogen testing; f...
158,MEDICAL CERT,SYSTEMIC_TOXICITY,systemic toxicity; acute systemic toxicity; ge...


In [58]:
med_cert_synonyms['SYNONYMS'] = med_cert_synonyms['SYNONYMS'].apply(lambda x : x.replace(',', ';'))
med_cert_synonyms

C:\Users\DSCMS5\AppData\Local\Temp\21\ipykernel_8852\856581444.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  med_cert_synonyms['SYNONYMS'] = med_cert_synonyms['SYNONYMS'].apply(lambda x : x.replace(',', ';'))


,TYPE,DEFINED_NAME,SYNONYMS
151,MEDICAL CERT,CYTOTOXICITY,cytotoxicity; cell toxicity; cellular toxicity...
152,MEDICAL CERT,DERMAL_IRRITATION,dermal irritation; skin irritation; skin irrit...
153,MEDICAL CERT,GENETOXICITY,genotoxicity; genetic toxicity; mutagenicity; ...
154,MEDICAL CERT,HEMOLYSIS,hemolysis; haemolysis; blood compatibility; re...
155,MEDICAL CERT,MUSCLE_IMPLANTATION,muscle implantation; intramuscular implantatio...
156,MEDICAL CERT,PHYSICOCHEMICAL_COMPLIANCE,physicochemical compliance; physical chemical ...
157,MEDICAL CERT,PYROGENICITY,pyrogenicity; pyrogen test; pyrogen testing; f...
158,MEDICAL CERT,SYSTEMIC_TOXICITY,systemic toxicity; acute systemic toxicity; ge...


In [59]:
med_certs = []
for idx, row in med_cert_synonyms.iterrows():
    cert = []
    defined_name = row['DEFINED_NAME'].lower()
    synonyms = row['SYNONYMS']
    if ";" in synonyms:
        synonym_list = [s.strip() for s in synonyms.split(';') if s.strip()]
    elif "," in synonyms:
        synonym_list = [s.strip() for s in synonyms.split(',') if s.strip()]
    else:
        synonym_list = [synonyms.strip()] if synonyms.strip() else []
    
    # med_cert_synonyms.at[idx, 'SYNONYM_LIST'] = synonym_list
    cert.append(defined_name)
    cert.append(synonym_list)
    med_certs.append(cert)
med_certs

[['cytotoxicity',
  ['cytotoxicity',
   'cell toxicity',
   'cellular toxicity',
   'cell viability',
   'cell viability testing',
   'in vitro cytotoxicity',
   'cell compatibility',
   'biocompatibility cytotoxicity',
   'toxicity to cells',
   'cell culture toxicity',
   'cyto',
   'cytotox']],
 ['dermal_irritation',
  ['dermal irritation',
   'skin irritation',
   'skin irritation testing',
   'cutaneous irritation',
   'skin compatibility',
   'skin contact safety',
   'irritation potential',
   'dermatological irritation',
   'primary skin irritation',
   'skin reactivity',
   'intracutaneous reactivity',
   'intracutaneous',
   'intracutaenous test']],
 ['genetoxicity',
  ['genotoxicity',
   'genetic toxicity',
   'mutagenicity',
   'mutagenic potential',
   'dna damage assessment',
   'genetic damage testing',
   'ames test compliance',
   'chromosomal aberration testing',
   'genotoxic safety',
   'gene tox',
   'geno',
   'ames']],
 ['hemolysis',
  ['hemolysis',
   'haemolysi

In [60]:
# Remove grades names from the color code grade names if it does not start with any of the actual brands in list
all_brands_list = []  
for key, values in brand_synonyms.items():  
    all_brands_list.append(key)  
    all_brands_list.extend(values)  


ce_grades_with_color_code = [grade for grade in ce_grades_with_color_code if any(grade.startswith(item) for item in all_brands_list)]
ce_grades_with_color_code = [i for i in ce_grades_with_color_code if "olultramid" not in i]
ce_grades_with_color_code = [i for i in ce_grades_with_color_code if "basf" not in i]
len(ce_grades_with_color_code)

404973

In [61]:
# ce_grades_with_color_code_new = [grade for grade in ce_grades_with_color_code if any(grade.startswith(item) for item in all_brands_list)]
# items_to_be_removed_from_unique_list = set(ce_grades_with_color_code) - set(ce_grades_with_color_code_new)
# normalized_unique_values_current["GRADE"] = list(set(normalized_unique_values_current["GRADE"])-items_to_be_removed_from_unique_list)

In [62]:
len(normalized_unique_values_current["GRADE"])

21202

In [63]:
# {'ecomid': 'ecomid', 'frianyl': 'frianyl', 'gur': 'gur', 'rynite': 'rynite'}

In [64]:
def normalize_query(s):
    # Remove special characters and convert to lower case
    return re.sub(r'\W+', '', s).lower()

In [65]:
unique_values.keys()

dict_keys(['BRAND', 'POLYMER', 'PROPERTY', 'FEATURE', 'FILLER', 'GRADE', 'CERTIFICATION', 'COMPETITOR_GRADE', 'APPLICATION', 'MODIFIER', 'UNIT', 'FILLER_PERCENTAGE', 'GRADE_WITHOUT_BRAND', 'COMPETITOR_GRADE_TRANSFORMED', 'COMP_GRADE_WITHOUT_BRAND', 'COMP_GRADE_TRANSFORMED_WITHOUT_BRAND'])

In [66]:
competitor_grades = read_data_from_snowflake_table(cur,"select * from competitor_data")
competitor_grades = list(set(competitor_grades['competitor_grade']))

In [67]:
normalized_values = {}
normalized_values['columnstoIgnore'] = []
normalized_values['GRADE'] = list(set([normalize_string(x) for x in unique_values['GRADE']] + [normalize_string(x) for x in unique_values['GRADE_WITHOUT_BRAND']] + [normalize_string(x) for x in new_grade_names]))
normalized_values['COMPETITOR_GRADE'] = list(set([normalize_string(x) for x in unique_values['COMPETITOR_GRADE']] + \
    [normalize_string(x) for x in unique_values['COMP_GRADE_WITHOUT_BRAND']] + \
    [normalize_string(x) for x in unique_values['COMPETITOR_GRADE_TRANSFORMED']] + \
    [normalize_string(x) for x in unique_values['COMP_GRADE_TRANSFORMED_WITHOUT_BRAND']]+\
    [normalize_string(x) for x in competitor_grades]))
    
normalized_values['columnstoIgnore'].extend([normalize_string(x) for x in unique_values['BRAND']])
normalized_values['columnstoIgnore'].extend([normalize_string(x) for x in unique_values['POLYMER']])
normalized_values['columnstoIgnore'].extend([normalize_string(x) for x in unique_values['FILLER']])
normalized_values['columnstoIgnore'].extend(outOfScopeData['brands'])
normalized_values['columnstoIgnore'] = list(set(normalized_values['columnstoIgnore']))
normalized_values['columnsforSubstringCheck'] = list(set([normalize_string(x) for x in unique_values['FEATURE']] + [normalize_string(x) for x in unique_values['PROPERTY']] + [normalize_string(x) for x in all_features]))

In [68]:
normalized_values['BRAND'] = [normalize_string(x) for x in unique_values['BRAND']]

In [69]:
for brand in spt_brands:
    if brand not in normalized_values['BRAND']:
        print(brand)
        normalized_values['BRAND'].append(brand)

normalized_values['BRAND'].sort()

minlon
talcoprene
neolast
pipelon
at
vitaldose
clarifoil


In [70]:
normalized_values['BRAND']

['at',
 'ateva',
 'celanex',
 'celanyl',
 'celcon',
 'celstran',
 'celstranlft',
 'clarifoil',
 'cn',
 'cnl',
 'coolpoly',
 'cp',
 'crastin',
 'cs',
 'cslft',
 'cx',
 'eco',
 'ecomid',
 'elvamide',
 'ff',
 'fo',
 'forflex',
 'fortron',
 'fp',
 'frianyl',
 'geolast',
 'geolast',
 'gur',
 'hf',
 'hostaform',
 'hytrel',
 'impet',
 'kep',
 'kepital',
 'laprene',
 'lcpa',
 'lftcelstran',
 'litepol',
 'lp',
 'minlon',
 'neolast',
 'omnipro',
 'omnitech',
 'pb',
 'pibiter',
 'pipelon',
 'po',
 'polifor',
 'rynite',
 'santoprene',
 'sofprene',
 'sofpur',
 'stp',
 'talcoprene',
 'tc',
 'tecnoprene',
 'thermx',
 'tp',
 'tx',
 'va',
 'vamac',
 'vandar',
 've',
 'vectra',
 'vitaldose',
 'ze',
 'zenite',
 'zytel',
 'zytelhtn',
 'zytellcpa',
 'zytelplsxt']

In [71]:
normalized_values.keys()

dict_keys(['columnstoIgnore', 'GRADE', 'COMPETITOR_GRADE', 'columnsforSubstringCheck', 'BRAND'])

In [72]:
region_cert_values_to_ignore_from_grade = ["china","asia","europe","america","ford","lfrt","impact72","industrial","bmw","ktw"]

In [73]:
applications_values_to_ignore_from_grade = ["pv", "hv", "pcb", "ccm", "mcb", "w&c", "tb", "usb", "sim", "lv", "labs", "smt", "fpc", "vcm", "rf", "ofc", "pc", "ff", "ac", "btb", "cmc", "bev", "wtb", "cpu", "ltcc", "vr", "htls", "eps", "rar", "lcd", "cgm", "iot"]

In [74]:
normalized_values['GRADE'] = list(set(normalized_values['GRADE'] + outOfScopeDataGrades + normalized_unique_values_current['GRADE']))
normalized_values['COMPETITOR_GRADE'] = list(set(normalized_values['COMPETITOR_GRADE'] + normalized_unique_values_current['COMPETITOR_GRADE']))
normalized_values['columnstoIgnore'] = list(set(normalized_values['columnstoIgnore'] + normalized_unique_values_current['columnstoIgnore'] + region_cert_values_to_ignore_from_grade + applications_values_to_ignore_from_grade))
normalized_values['columnsforSubstringCheck'] = list(set(normalized_values['columnsforSubstringCheck'] + normalized_unique_values_current['columnsforSubstringCheck']))

In [75]:
normalized_values['GRADE'] = list(set(normalized_values['GRADE'] + [normalize_string(i) for i in ce_grades])) # + ce_grades_with_color_code
normalized_values['COMPETITOR_GRADE'] = [i for i in normalized_values['COMPETITOR_GRADE'] if len(i)>2]

In [76]:
normalized_values['FEATURE'] = feature_list

In [77]:
normalized_values['MEDICAL_CERT'] = med_certs
normalized_values['CHEMICAL_RESISTANCE'] = valid_format_nested

In [ ]:
# validating the list of ce grades present (since for bug 203909 minlon grades were not populated in unique value list)
# df_ce_grades_normalized = pd.DataFrame(normalized_values['GRADE'])
# df_ce_grades_normalized.to_csv('celanese_grades__normalized_list.csv', index=False, header=False)

In [78]:
len(normalized_values['GRADE']), len(normalized_values['COMPETITOR_GRADE'])

(21822, 56885)

In [79]:
len(normalized_values['FEATURE'])

41

In [80]:
normalized_values["GRADE"] = [grade for grade in normalized_values["GRADE"] if any(grade.startswith(item) for item in all_brands_list)]
normalized_values["GRADE"] = [grade for grade in normalized_values["GRADE"] if "basf" not in grade]
len(normalized_values['GRADE'])


21225

In [ ]:
# #all_brands_list
# import csv
# # validating the list of ce grades present (since for bug 203909 minlon grades were not populated in unique value list)
# csv_filename = "brand_synonyms.csv"
# fieldnames = ["Brands", "Brand_Synonyms"]
# with open(csv_filename, mode='w', newline='') as file:
#     writer = csv.DictWriter(file, fieldnames=fieldnames)
#     writer.writeheader()
#     for brand, synonyms in brand_synonyms.items():
#         for synonym in synonyms:
#                 writer.writerow({"Brands": brand, "Brand_Synonyms": synonym})
# # df_brand_synonyms = pd.DataFrame(brand_synonyms)
# # df_brand_synonyms.to_csv('brand_synonyms.csv', index=False, header=False)

In [81]:
[i for i in ce_grades_with_color_code if "basf" in i]

[]

In [82]:
[i for i in normalized_values["GRADE"] if "nw02" in i]

[]

In [83]:
[i for i in ce_grades_with_color_code if "olultra" in i]

['colultramarineviolet525kgbag',
 'colultramarineblep3825kgbag',
 'colultramarinebl08',
 'colultramarinebl08hollidaypigments']

## Certification

In [84]:
import requests

def get_dev_unique_values():
    url = "https://apim-gst-dev.azure-api.net/func-uiproperties-d-ussc-01/func-ui-properties?userType=internal"
    
    headers = {
        "Content-Type": "application/json",
        "Ocp-Apim-Subscription-Key": "c2404e96bf0b471daf7f2b1091094fa1"
              }
    try:
        res = requests.post(url, headers=headers)
        res = res.json()
    except Exception as e:  
        print(e)
        
    return res

uniue_values = get_dev_unique_values()
print(uniue_values.keys())
uniue_values['results'].keys()

dict_keys(['success', 'status', 'results'])


dict_keys(['Property', 'Chemical_Resistance', 'Feature', 'UL', 'Certification', 'Auto_Approval', 'Brand', 'Polymer', 'Filler', 'Market', 'Industry_Group'])

In [85]:
all_auto_certs = []
for cert_data in uniue_values['results']['Auto_Approval']:
    for cert in cert_data['CERTIFICATIONS']:
        if cert_data['OEM_Name'].lower() == 'santoprene':
            print('found non oem')
        else:
            all_auto_certs.append((cert['CERTIFICATION'].lower(), cert_data['OEM_Name'].lower()))

all_auto_certs = list(set(all_auto_certs))
all_auto_certs

[('tl 522 81-b tpc-et', 'vw group'),
 ('gmw16733p-pbt-gf15', 'general motors (gm)'),
 ('ms-db-570 / cpn-3502', 'stellantis-chrysler'),
 ('ms.500017 / pa66.2810f.6i.hs', 'stellantis'),
 ('ms-db-21 / cpn-3608', 'stellantis-chrysler'),
 ('tl 524 40-a pa6-gf50', 'vw group'),
 ('dbl5405.05 pa66 gf35', 'mercedes-benz'),
 ('vw 50136', 'vw group'),
 ('tl 520 62 pa-gf35', 'vw group'),
 ('gmw17457p-tpc-et-type 2', 'general motors (gm)'),
 ('b62 0300 / 61/u4/225e/217m/c2b/c4', 'stellantis'),
 ('ms.50017 / cpn-4367', 'stellantis-chrysler'),
 ('b62 0300 / 61/u4/223e/211m/c2b/c4', 'stellantis'),
 ('dbl5403 pbt-i-gf15', 'mercedes-benz'),
 ('b62 0300 / 61/210m+/215e+/c1b', 'stellantis'),
 ('n-004-0047', 'nio'),
 ('tst n 055 54.37', 'continental'),
 ('ms.50095 / cpn-1986', 'stellantis-chrysler'),
 ('vw 50133 pa66-7-a', 'vw group'),
 ('ms.50103 / cpn-5292', 'stellantis-chrysler'),
 ('wsa-m4d768-a', 'ford'),
 ('jf03-26', 'faw group'),
 ('ms.50017 / cpn-2224', 'stellantis-chrysler'),
 ('gmw15890p-pp-gf60'

In [86]:
normalized_values['AUTO_CERT'] = [(normalize_string(x[0]), x[1]) for x in all_auto_certs]

In [87]:
normalized_values["GRADE"] = [i for i in normalized_values["GRADE"] if "amodel" not in i]

In [88]:
competitor_data2 = read_data_from_snowflake_table(cur,"select * from ANALYTICS_DEV.gst_curated.comp_offset_json_property_mapping")
# QA Snowflake connection
# competitor_data2 = read_data_from_snowflake_table(cur,"select * from ANALYTICS_QA.gst_curated.comp_offset_json_property_mapping")
competitor_data2.head()

,competitor,grade_name,feature,spt,property_id,property_name,value,unit,id,abbt,filler,polymer,grade_moisture_dependent,test_method,filler_load,filler_description,brand,competitor_old,brand_old,grade_name_old
0,MAIP S.R.L.,HIPRO CP FV30W U2,False,True,11,"Tensile stress at break, 5mm/min",50,MPa,10500197,PP-GF,GF,PP,False,ISO 527-1/-2,None,Glass Fiber,HIPRO,MAIP,HIPRO,HIPRO CP FV30W U2
1,SHPP Global Technologies,NORYL™ Resin NH5120RC3,False,True,11,"Tensile stress at break, 5mm/min",48,MPa,10475798,(PPE+PS),,"PPE,PS",False,ISO 527-1/-2,None,None,NORYL™,SABIC,NORYL™,NORYL™ Resin NH5120RC3
2,SHPP Global Technologies,LNP™ STAT-KON™ Compound DSL229,False,True,783,Tensile modulus,2450,MPa,10472991,(PC+PTFE),,"PC,PTFE",False,ASTM D 638,None,None,LNP™,SABIC,LNP™,LNP™ STAT-KON™ Compound DSL229
3,Uteksol,Solplast TH 60A9024 B,False,True,173,Shore A hardness 3s,60,,10514264,TPS,,TPS,False,ISO 48-4 / ISO 868,None,None,Solplast,Uteksol,Solplast,Solplast TH 60A9024 B
4,Kraiburg TPE,THERMOLAST® K TC7PRZ,True,False,1,Injection Molding,True,,10487316,TPE,,TPE,False,None,None,None,THERMOLAST®,KRAIBURG TPE,THERMOLAST®,THERMOLAST® K TC7PRZ


In [89]:
competitor_brands = competitor_data2['brand'].unique().tolist() + competitor_data2['brand_old'].unique().tolist()
competitor_brands = [x for x in competitor_brands if ('®' in x or '™' in x)]
# competitor_brands = list(set([x.strip(" ®™.-/,").lower() for x in competitor_brands if x]))
competitor_brands[:10]

['NORYL™',
 'LNP™',
 'THERMOLAST®',
 'XANTAR™',
 'Makrolon®',
 'BERGAMID®',
 'ExxonMobil™',
 'EcoLene™',
 'Tritan™',
 'Nypol®']

In [90]:
len(competitor_brands)

1688

In [91]:
for x in competitor_brands:
    x2 = x.strip(" ®™.-/,").lower()
    if ('®' in x2 or '™' in x2):
        print(x2)

tenac™-c
slovamid®66
slovamid®6
sabic® sabital
slovamid®66/6
p84®nt1
victrex® ht
p84®uht
sabic® stamax
victrex® st
victrex® fg
slovamid®6/66
victrex® wg
p84®nt2
sabic® supeer
slovamid®6/6t
emac®+
ebac®+
sabic® fortify
tenac™-c
slovamid®66
slovamid®6
slovamid®66/6
p84®nt1
p84®uht
slovamid®6/66
p84®nt2
slovamid®6/6t
emac®+
ebac®+


In [92]:
for i in competitor_brands:
    if len(i) < 5:
        print(i)

LNP™
DOW™
TPX®
NAS®
iOn™
XT®
LNP™
DOW™
TPX®
NAS®
iOn™
XT®


In [93]:
additional_competitor_brands = [
    'slovamid',
    'umg',
    'p84',
    'tenac',
    'eva', # also a OOS polymer

]
normalized_competitor_brands = [normalize_string(x) for x in competitor_brands+additional_competitor_brands]
normalized_competitor_brands = list(set(normalized_competitor_brands))
normalized_competitor_brands

['titanlene',
 'apinat',
 'surpass',
 'onflexv',
 'tisarbon',
 'exceed',
 'tecothane',
 'enviramid',
 'hipex',
 'valtra',
 'apel',
 'armovil',
 'riblene',
 'kresin',
 '3dxpro',
 'calibre',
 'starpet',
 'tenite',
 'daploy',
 'multilon',
 'solvadura',
 'palran',
 'exelene',
 'proflow',
 'alkamax',
 'eastalite',
 'compadur',
 'desmovit',
 'xantar',
 'ixef',
 'sinterline',
 'forstir',
 'tisaform',
 'formolon',
 'starmed',
 'saxalac',
 'sumipex',
 'vylopet',
 'kopla',
 'palstyrol',
 'neostar',
 'onflex',
 'ecopaxx',
 'technaset',
 'advance400',
 'kopa',
 'thermolast',
 'gravitech',
 'aurum',
 'borshape',
 'duragrip',
 'inspire',
 'tisblend',
 'achieve',
 'polyflam',
 'laser',
 'sumikaexcel',
 'agility',
 'bayblend',
 'ecodear',
 'niblend',
 'ecozen',
 'gitop',
 'clearlux',
 'chronoprene',
 'solef',
 'forlene',
 'stirolan',
 'isostyr',
 'bergadur',
 'saxatec',
 'vestakeep',
 'hiplex',
 'koolymer',
 'copec',
 'alkatane',
 'umgabs',
 'evathene',
 'naxell',
 'lnp',
 'isoplast',
 'sicoklar',
 'e

In [94]:
for i in normalized_competitor_brands:
    if len(i) < 5:
        print(i)


apel
ixef
kopa
lnp
4pet
tpx
4max
udea
nas
upes
eva
echo
soe
emac
umg
cet
4loy
4lex
4pom
ion
utec
icom
udel
xt
eval
ixan
4lac
4mid
ebac
apec
dow
xtel
p84
queo
4dur


In [95]:
normalized_values['COMPETITOR_BRAND'] = normalized_competitor_brands
normalized_values.keys()

dict_keys(['columnstoIgnore', 'GRADE', 'COMPETITOR_GRADE', 'columnsforSubstringCheck', 'BRAND', 'FEATURE', 'MEDICAL_CERT', 'CHEMICAL_RESISTANCE', 'AUTO_CERT', 'COMPETITOR_BRAND'])

In [96]:
with open("../dependencies/normalized_unique_values_for_grade_mapping.json", "w") as fp:
    json.dump(normalized_values, fp)

## Competitor names

In [97]:
all_competitor_names_current = json.load(open("../dependencies/normalized_competitor_names.json"))

In [98]:
competitor_data = read_data_from_snowflake_table(cur,"select * from competitor_data")
competitor_names = competitor_data['competitor'].unique().tolist()

In [99]:

additional_competitor_data = read_data_from_snowflake_table(cur,"select * from COMPETITIVE_LEGAL_BRAND_PRODUCER_MAPPING")
additional_competitor_names = additional_competitor_data['mdc_competitor_name'].unique().tolist()
for item in additional_competitor_data['alternative_competitor_name']:
    if item!=None:
        additional_competitor_names.extend(item.split(";"))
additional_competitor_names = list(set(additional_competitor_names))

In [100]:
competitor_names = list(set(competitor_names+additional_competitor_names))

In [101]:
sample_comp_grades = [i.lower() for i in competitor_data['competitor_grade'].unique()]

In [102]:
def normalize_comp_names(comp_name):
    splitted_competitor_names = []
    if comp_name is not None: #fix for AttributeError: 'NoneType' object has no attribute 'lower'
        comp_name = comp_name.lower()
    splitted_competitor_names.append(comp_name)
    comp_name_split1 = []
    if comp_name is not None:
        comp_name_split1 = comp_name.split()
    comp_name_split2=[]
    for item in comp_name_split1:
        comp_name_split2.extend(item.split("_"))
    comp_name_split3=[]
    for item in comp_name_split2:
        comp_name_split3.extend(item.split("-"))
    comp_name_split4=[]
    for item in comp_name_split3:
        comp_name_split4.extend(item.split(","))
    comp_name_split5=[]
    for item in comp_name_split4:
        comp_name_split5.extend(item.split("&"))
    splitted_competitor_names.extend(comp_name_split5)
    return splitted_competitor_names


In [103]:
print(sum(x is None for x in competitor_names))


none_indices = [i for i, x in enumerate(competitor_names) if x is None]
print(none_indices)

print(len(competitor_names))

competitor_names = [x for x in competitor_names if x is not None] #handling all errors related to NoneType

print(len(competitor_names))


1
[229]
748
747


In [104]:
all_competitor_names = []
for comp_name in competitor_names:
    all_competitor_names.extend(normalize_comp_names(comp_name))
all_competitor_names=list(set(all_competitor_names))

all_competitor_names = [i for i in all_competitor_names if len(i)>2] #fix for TypeError: object of type 'NoneType' has no len()

In [105]:

# Create a new list to hold the filtered elements from 'a'  
filtered_a = []  
  
for item_a in all_competitor_names:  
    # Assume initially that the item is not a substring of any item in 'b'  
    is_substring = False  
    for item_b in sample_comp_grades:  
        if item_b.startswith(item_a) or item_b.endswith(item_a):  
            # If 'item_a' is found to be a substring of 'item_b', update the flag and break out of the loop  
            is_substring = True  
            print(item_a)
            break  
    # If 'item_a' is not a substring of any items in 'b', add it to the filtered list  
    if not is_substring:  
        filtered_a.append(item_a)  
  
# Optionally, if you need 'a' to be the list without the substrings, you can assign 'filtered_a' back to 'a'  
# all_competitor_names = filtered_a  
  
# print(all_competitor_names)  


polyram
vamp
toray
kopla
ferro
dawn
taro
tech
akro
nylene
tasnee
epitec
nanovia
prime
the
jsr
chem
lapex
polymer
color
eastman
matrix
tpe
invista
comp
inno
petro
americas
lati
nova
polyblend
bada
biofibre
witcom
api
fabru
titan
pet
abs
prl
delrin
volaprint
hip
rsh
rtp
ram
ube
exxonmobil
elasto
sax
grivory
bond
ria
bio
sipol
star
pal
dow
elastron
plast
dic
ter
kimya
green
techno
top
europe
sabic
epi
total


In [106]:
comp_names_to_remove = ["witcom",
                        "kopla",
                        "victrex",
                        "prl",
                        "volaprint",
                        "nylene",
                    "grivory",
                    "delrin",
                    "rtp",
                    "kingfa",
                    "victrex"]

In [107]:
all_competitor_names = list(set(all_competitor_names+all_competitor_names_current))
all_competitor_names = [i for i in all_competitor_names if i not in comp_names_to_remove]

In [108]:
[i for i in all_competitor_names if i.startswith("delrin")]

['delrin usa, llc', 'delrin usa llc']

In [109]:

with open("../dependencies/normalized_competitor_names.json", "w") as fp:
    json.dump(all_competitor_names , fp)

In [ ]:
text = "mitsubishi chemicals co. delrin ultrdur xyz"
text_words = text.split()
print(text_words)
# Remove words from the start  
while text_words and text_words[0].lower() in all_competitor_names: 
    print(text_words[0].lower()) 
    text_words.pop(0)
    print(text_words)
# Remove words from the end  
while text_words and text_words[-1].lower() in all_competitor_names:  
    text_words.pop() 

' '.join(text_words).strip()

['mitsubishi', 'chemicals', 'co.', 'delrin', 'ultrdur', 'xyz']
mitsubishi
['chemicals', 'co.', 'delrin', 'ultrdur', 'xyz']
chemicals
['co.', 'delrin', 'ultrdur', 'xyz']
co.
['delrin', 'ultrdur', 'xyz']


'delrin ultrdur xyz'

In [ ]:
"mitsubishi" in all_competitor_names

True

In [ ]:
# validating the list of ce grades present (since for bug 203909 minlon grades were not populated in unique value list)
# df_ce_grades_normalized_new = pd.DataFrame(normalized_values['GRADE'])
# df_ce_grades_normalized_new.to_csv('celanese_grades__normalized_list_new.csv', index=False, header=False)

## Test

In [ ]:
%%time
query_for_grade_match = normalize_string("N 2770")
process.extract(query_for_grade_match, normalized_values['COMPETITOR_GRADE'])

CPU times: total: 359 ms
Wall time: 416 ms


[('n2770k', 91),
 ('ultraformn2770k', 90),
 ('ultraformn2770kat', 90),
 ('n2770kat', 90),
 ('revolven270', 80)]

In [ ]:
%%time
query_for_grade_match = normalize_string("h100")
process.extract(query_for_grade_match, normalized_values['GRADE'])

CPU times: total: 125 ms
Wall time: 151 ms


[('keph100', 90),
 ('neolasth100nc010', 90),
 ('neolasth100', 90),
 ('neoh100nc010', 90),
 ('kepitalh100', 90)]

In [ ]:
query_for_grade_match

'h100'

In [ ]:
unique_values.keys()

dict_keys(['BRAND', 'POLYMER', 'PROPERTY', 'FEATURE', 'FILLER', 'GRADE', 'CERTIFICATION', 'COMPETITOR_GRADE', 'APPLICATION', 'MODIFIER', 'UNIT', 'FILLER_PERCENTAGE', 'GRADE_WITHOUT_BRAND', 'COMPETITOR_GRADE_TRANSFORMED', 'COMP_GRADE_WITHOUT_BRAND', 'COMP_GRADE_TRANSFORMED_WITHOUT_BRAND'])

In [ ]:
for key in ['COMPETITOR_GRADE','BRAND', 'POLYMER', 'PROPERTY', 'FEATURE', 'FILLER', 'CERTIFICATION', 'APPLICATION', 'MODIFIER', 'UNIT', 'FILLER_PERCENTAGE']:
    
    print(key)
    for item in unique_values[key]:
        query_for_grade_match = normalize_string(item)
        if query_for_grade_match not in normalized_values['columnstoIgnore'] and not re.fullmatch(r'(\d+gf|gf\d+|\d+mf|mf\d+)', query_for_grade_match) and len(query_for_grade_match)>3:
            match = process.extract(query_for_grade_match, normalized_values['GRADE'], limit=1)
            if match[0][1]>=90:
                print(query_for_grade_match,": ", match[0][0])
    print("\n\n")
    break

COMPETITOR_GRADE
5220b :  rynitere5220bk503
adstifha1152 :  a115
akroloypaicf20black5268 :  icf
akroloypaicf30black5269 :  icf


KeyboardInterrupt: 

In [ ]:
for key in ['BRAND', 'POLYMER', 'PROPERTY', 'FEATURE', 'FILLER', 'CERTIFICATION', 'APPLICATION', 'MODIFIER', 'UNIT', 'FILLER_PERCENTAGE']:
    print(key)
    for item in unique_values[key]:
        query_for_grade_match = normalize_string(item)
        if query_for_grade_match not in normalized_values['columnstoIgnore'] and not re.fullmatch(r'(\d+gf|gf\d+|\d+mf|mf\d+)', query_for_grade_match) and len(query_for_grade_match)>3:
            match = process.extract(query_for_grade_match, normalized_values['COMPETITOR_GRADE'], limit=1)
            if match[0][1]>=90:
                print(query_for_grade_match,": ", match[0][0])
    print("\n\n")

In [ ]:
re.fullmatch(r'(\d+gf|gf\d+|\d+mf|mf\d+)', "gf3000d")

In [ ]:
re.fullmatch(r'^(gf|mf)\d{2}$', "gf300")

In [ ]:
from elasticsearch import Elasticsearch, helpers

def get_es():
    ELASTIC_PASSWORD = "<ELASTICSEARCH_PASSWORD_DEV>"
    es = Elasticsearch(
        "https://elast-gst-nprd-ussc-01.es.privatelink.southcentralus.azure.elastic-cloud.com:9243",
        basic_auth=("elastic", ELASTIC_PASSWORD))
    print(es.info())
    return es

ES = get_es()
# fetch all ul properties
match_all_query = {
    "query": {
    "match_all": {}
    }
}

polymer_resp = ES.search(index='filter_polymer', body=match_all_query, size=100)
polymer_values = [hit['_source'] for hit in polymer_resp['hits']['hits']]